In [1]:
# Adicionem aqui as bibliotecas necessárias
import pandas as pd
import os
from PIL import Image
from torchvision import datasets, transforms, models
import cv2
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
path = '/content/drive/MyDrive/Projeto LIPAI/NDB-UFES'
dataset_name = "NDB-UFES"

In [4]:
import json
import os

def save_metrics_json(base_dir,
                     arch_name, mode, aug,
                     seed, acc, f1_macro, f1_weighted):

    dir_path = f'{base_dir}/{dataset_name}/results'
    os.makedirs(dir_path, exist_ok=True)

    results = {
        "architecture": arch_name,
        "mode": mode,
        "aug": aug,
        "seed": seed,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

    aug_str = "non_aug" if not aug else "aug"

    filename = f"{arch_name}_{mode}_{aug_str}_seed{seed}.json"
    path = os.path.join(dir_path, filename)

    with open(path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"Métricas salvas em: {path}")

# 1. Carregamento dos Dados

In [5]:
manifest = pd.read_csv(f'{path}/manifest.csv')
images = datasets.ImageFolder(root = f'{path}/dataset')

Divisão dos sets de acordo com o manifest

In [6]:
base_path = f'{path}/dataset/images'

def load_images_from_df(dataframe):
    images = []
    labels = []

    for idx, row in dataframe.iterrows():
        # Constrói o caminho completo: dataset/healthy/imagem.tif
        img_path = os.path.join(base_path, row['path'])

        # Leitura da imagem
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            labels.append(row['label_number'])
        else:
            print(f"Erro ao carregar: {img_path}")

    return np.array(images), np.array(labels)

# 3. Criar as divisões diretamente do CSV
train_df = manifest[manifest['sets'] == 'train']
val_df   = manifest[manifest['sets'] == 'val']
test_df  = manifest[manifest['sets'] == 'test']

# 4. Carregar os dados para as variáveis
print("Carregando imagens de treino...")
x_train, y_train = load_images_from_df(train_df)

print("Carregando imagens de validação...")
x_val, y_val = load_images_from_df(val_df)

print("Carregando imagens de teste...")
x_test, y_test = load_images_from_df(test_df)

# Conferindo os formatos
print("\nDivisão concluída:")
print(f"Treino: {x_train.shape}, {y_train.shape}")
print(f"Val:    {x_val.shape}, {y_val.shape}")
print(f"Teste:  {x_test.shape}, {y_test.shape}")

Carregando imagens de treino...
Carregando imagens de validação...
Carregando imagens de teste...

Divisão concluída:
Treino: (159, 1536, 2048, 3), (159,)
Val:    (39, 1536, 2048, 3), (39,)
Teste:  (39, 1536, 2048, 3), (39,)


## Transformações e Data Augmentation

Não queremos distorções brucas nas imagens histológicas, portanto, escolhemos aplicar apenas 3 operações geometricas.

* RandomHorizontalFlip: Espelhamento horizontal.
* RandomVerticalFlip: Espelhamento vertical.
* RandomRotation: Rotação suave de 15 graus.

In [7]:
transform_basic = transforms.Compose([ # Básica, sem augmentation
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

transform_augmentation = transforms.Compose([ # Com augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

## DataLoaders

Tamanho do Lote: 32

Estou ultilizando uma classe auxiliar HistologyDataset, que envolve os arrays numpy, e os alinha com o pipeline do torchvision.transforms.


In [8]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class HistologyDataset(Dataset):
    def __init__(self, images_array, labels_array, transform=None):
        self.images = images_array
        self.labels = labels_array
        self.transform = transform

    def __len__(self):
        # Tamanho total do dataset
        return len(self.labels)

    def __getitem__(self, idx):
        # Pega a imagem em formato Numpy
        img_np = self.images[idx]
        label = self.labels[idx]

        # Converte BGR para RGB
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

        # Converte o array para um objeto Imagem (PIL)
        img_pil = Image.fromarray(img_rgb)

        # Aplica as transformações
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)
        return img_tensor, label

In [9]:
BATCH_SIZE = 32

# Aplicação das transformações no treino.
train_dataset = HistologyDataset(x_train, y_train, transform=transform_augmentation)
val_dataset = HistologyDataset(x_val, y_val, transform=transform_basic)
test_dataset = HistologyDataset(x_test, y_test, transform=transform_basic)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders construídos!")
print(f"Total de lotes no treino: {len(train_loader)}")

DataLoaders construídos!
Total de lotes no treino: 5


# 2. Aplicação dos Algoritmos

Estou definindo uma função que permite aplicar os algoritmos de forma dinamica. Conforme as orientações, vamos usar:

* FS - From Scratch: Possui pesos aleatórios e aprende as caracteristicas visuais do zero.
* PT-ALL - Fine-tuning Completo: Possui pesos pré-treinados da ImageNet e será executado com todas as camadas treináveis.
* PT-FC - Backbone Congelado: Possui pesos pré-treinados da ImageNet e será executado com apenas o classificador treinável.

In [10]:
import timm
import torch.nn as nn
import numpy as np

def build_model(name, mode, num_classes):
    """Fábrica de modelos. """

    if mode == 'FS':
        model = timm.create_model(name, pretrained=False, num_classes=num_classes)
    elif mode == 'PT-ALL':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
    elif mode == 'PT-FC':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
        # Congela os parâmetros
        for param in model.parameters():
            param.requires_grad = False
        # Descongela os parâmetros da última camada
        for param in model.get_classifier().parameters():
            param.requires_grad = True
    else:
        raise ValueError(f"Modo de treinamento {mode} não encontrado.")
    return model

N = len(np.unique(y_train)) # Conta o número de classes únicas nos dados de treino

modelo_teste = build_model('convnextv2_atto', 'PT-FC', num_classes=N)

total_params = sum(p.numel() for p in modelo_teste.parameters())
trainable_params = sum(p.numel() for p in modelo_teste.parameters() if p.requires_grad)

print(f"Total de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis (PT-FC): {trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

Total de parâmetros: 3,388,363
Parâmetros treináveis (PT-FC): 963


# 3. Comparativos Globais



Irei criar uma dicionario history para guarda a acurácia e loss, do treino e e da validação.

In [11]:
# Para cada dataset, o grupo deverá apresentar comparações globais usando: <br>
#  * Gráfico de barras do F1-score por arquitetura, modo de treinamento e uso de augmentation. <br>
#  * Tabela com média e desvio padrão das repetições. <br>
#  * Gráfico com número de parâmetros por arquitetura. <br>
#  * Gráfico com complexidade computacional em GFLOPs considerando entrada 224 × 224.